In [0]:
# Provable Resilience Intelligence — ML Pipeline
# Lakelink Fiber: Span Risk Classification + Optical Anomaly Detection
# @ml-engineer | Project: provable_resilience_intelligence

%pip install xgboost psycopg[binary] --quiet

In [0]:
%sql
-- ============================================================
-- P2-RISK-MODEL: Span Risk Classification Feature Engineering
-- Target: Predict service-affecting events per logical resource
-- Sources: alarm (100K), resource_performance (10K), logical_resource (100K)
-- ============================================================

CREATE OR REPLACE TEMP VIEW alarm_features AS
SELECT 
    logical_resource_id,
    COUNT(*) as alarm_count,
    SUM(CASE WHEN perceived_severity = 'critical' THEN 1 ELSE 0 END) as critical_alarm_count,
    SUM(CASE WHEN perceived_severity = 'major' THEN 1 ELSE 0 END) as major_alarm_count,
    SUM(CASE WHEN perceived_severity = 'minor' THEN 1 ELSE 0 END) as minor_alarm_count,
    SUM(CASE WHEN perceived_severity = 'warning' THEN 1 ELSE 0 END) as warning_alarm_count,
    SUM(CASE WHEN is_root_cause = true THEN 1 ELSE 0 END) as root_cause_count,
    SUM(CASE WHEN is_crossed_threshold = true THEN 1 ELSE 0 END) as threshold_crossing_count,
    AVG(CAST(observed_metric_value AS DOUBLE)) as avg_observed_metric,
    AVG(CAST(threshold_value AS DOUBLE)) as avg_threshold_value,
    COUNT(DISTINCT technology_domain) as tech_domain_diversity,
    MAX(CASE WHEN service_affecting = 'service_affecting' THEN 1 ELSE 0 END) as has_service_affecting_alarm
FROM cdm_tmforum.tmf_resource.alarm
WHERE logical_resource_id IS NOT NULL
GROUP BY logical_resource_id;

CREATE OR REPLACE TEMP VIEW perf_features AS
SELECT 
    logical_resource_id,
    AVG(CAST(availability_percent AS DOUBLE)) as avg_availability,
    AVG(CAST(utilization_percent AS DOUBLE)) as avg_utilization,
    AVG(CAST(error_count AS DOUBLE)) as avg_error_count,
    AVG(CAST(indicator_value AS DOUBLE)) as avg_indicator_value,
    SUM(CASE WHEN threshold_crossing_flag = true THEN 1 ELSE 0 END) as perf_threshold_crossings,
    SUM(CASE WHEN sla_breach_flag = true THEN 1 ELSE 0 END) as sla_breach_count,
    COUNT(DISTINCT technology_domain) as perf_tech_domains
FROM cdm_tmforum.tmf_resource.resource_performance
WHERE logical_resource_id IS NOT NULL
GROUP BY logical_resource_id;

SELECT 'Feature views created' as status

In [0]:
import mlflow
import mlflow.xgboost
import xgboost as xgb
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, confusion_matrix

mlflow.set_experiment("/Users/stephen.hage@databricks.com/provable-resilience-intelligence/span_risk_classification")

# Build feature matrix
resource_features = spark.sql("""
    SELECT logical_resource_id,
           CAST(bandwidth_mbps AS DOUBLE) as bandwidth_mbps,
           DATEDIFF(CURRENT_DATE(), activation_date) as resource_age_days,
           CASE administrative_state 
               WHEN 'unlocked' THEN 0 WHEN 'locked' THEN 1 WHEN 'shuttingDown' THEN 2 ELSE 3 
           END as admin_state_encoded
    FROM cdm_tmforum.tmf_resource.logical_resource
""")

features_df = (
    resource_features
    .join(spark.table("alarm_features"), "logical_resource_id", "left")
    .join(spark.table("perf_features"), "logical_resource_id", "left")
    .fillna(0)
    .withColumn("label", F.when(F.col("has_service_affecting_alarm") == 1, 1).otherwise(0))
)

feature_cols = [
    "alarm_count", "critical_alarm_count", "major_alarm_count", "minor_alarm_count",
    "warning_alarm_count", "root_cause_count", "threshold_crossing_count",
    "avg_observed_metric", "avg_threshold_value", "tech_domain_diversity",
    "avg_availability", "avg_utilization", "avg_error_count", "avg_indicator_value",
    "perf_threshold_crossings", "sla_breach_count", "perf_tech_domains",
    "bandwidth_mbps", "resource_age_days", "admin_state_encoded"
]

pdf = features_df.select(feature_cols + ["label", "logical_resource_id"]).toPandas()
X = pdf[feature_cols].values
y = pdf["label"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

with mlflow.start_run(run_name="span_risk_xgboost_production") as run:
    params = {
        "objective": "binary:logistic", "eval_metric": "aucpr",
        "max_depth": 6, "learning_rate": 0.05, "n_estimators": 300,
        "scale_pos_weight": scale_pos_weight, "subsample": 0.8,
        "colsample_bytree": 0.8, "min_child_weight": 5,
        "reg_alpha": 0.1, "reg_lambda": 1.0, "random_state": 42,
    }
    mlflow.log_params(params)
    mlflow.set_tag("project", "provable_resilience_intelligence")
    mlflow.set_tag("model_type", "span_risk_classification")
    mlflow.set_tag("brand", "Lakelink Fiber")
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    roc_auc = roc_auc_score(y_test, y_prob)
    avg_precision = average_precision_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("avg_precision", avg_precision)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("precision", cm[1,1]/(cm[1,1]+cm[0,1]) if (cm[1,1]+cm[0,1])>0 else 0)
    mlflow.log_metric("recall", cm[1,1]/(cm[1,1]+cm[1,0]) if (cm[1,1]+cm[1,0])>0 else 0)
    
    mlflow.xgboost.log_model(model, artifact_path="span_risk_model", input_example=X_test[:3])
    
    importance = sorted(zip(feature_cols, model.feature_importances_), key=lambda x: x[1], reverse=True)
    print(f"ROC-AUC: {roc_auc:.4f} | PR-AUC: {avg_precision:.4f} | F1: {f1:.4f}")
    print(f"Confusion: TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}")
    print("\nTop features:")
    for feat, imp in importance[:10]:
        print(f"  {feat}: {imp:.4f}")

In [0]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

mlflow.set_experiment("/Users/stephen.hage@databricks.com/provable-resilience-intelligence/optical_anomaly_detection")

perf_pdf = spark.sql("""
    SELECT resource_performance_id, logical_resource_id,
           CAST(availability_percent AS DOUBLE) as availability_pct,
           CAST(utilization_percent AS DOUBLE) as utilization_pct,
           CAST(error_count AS DOUBLE) as error_count,
           CAST(indicator_value AS DOUBLE) as indicator_value,
           CAST(max_value AS DOUBLE) as max_value,
           CAST(min_value AS DOUBLE) as min_value,
           CASE WHEN threshold_crossing_flag = true THEN 1 ELSE 0 END as threshold_crossed,
           CASE WHEN sla_breach_flag = true THEN 1 ELSE 0 END as sla_breached,
           CAST(threshold_value AS DOUBLE) as threshold_value,
           technology_domain, severity
    FROM cdm_tmforum.tmf_resource.resource_performance
""").toPandas()

anom_features = ["availability_pct", "utilization_pct", "error_count", "indicator_value",
                 "max_value", "min_value", "threshold_crossed", "sla_breached", "threshold_value"]

X_anom = perf_pdf[anom_features].fillna(0).values
scaler = StandardScaler()
X_anom_scaled = scaler.fit_transform(X_anom)

with mlflow.start_run(run_name="optical_anomaly_iforest_production") as anom_run:
    iso_params = {"n_estimators": 200, "contamination": 0.05, "max_features": 0.8, "random_state": 42}
    mlflow.log_params(iso_params)
    mlflow.set_tag("project", "provable_resilience_intelligence")
    mlflow.set_tag("model_type", "optical_anomaly_detection")
    mlflow.set_tag("brand", "Lakelink Fiber")
    
    iso_forest = IsolationForest(**iso_params)
    iso_forest.fit(X_anom_scaled)
    
    predictions = iso_forest.predict(X_anom_scaled)
    scores = iso_forest.decision_function(X_anom_scaled)
    perf_pdf["anomaly_label"] = predictions
    perf_pdf["anomaly_score"] = scores
    
    n_anom = (predictions == -1).sum()
    mlflow.log_metric("n_anomalies", n_anom)
    mlflow.log_metric("anomaly_rate", n_anom / len(predictions))
    mlflow.sklearn.log_model(iso_forest, artifact_path="anomaly_model")
    
    # Validation
    anom_mask = predictions == -1
    print(f"Anomalies: {n_anom}/{len(predictions)} ({100*n_anom/len(predictions):.1f}%)")
    print(f"Threshold crossing rate: anomaly={perf_pdf.loc[anom_mask, 'threshold_crossed'].mean():.3f} vs normal={perf_pdf.loc[~anom_mask, 'threshold_crossed'].mean():.3f}")
    print(f"\nAnomalies by technology domain:")
    print(perf_pdf[anom_mask].groupby('technology_domain').size().sort_values(ascending=False).head(10).to_string())

In [0]:
# =============================================================
# Summary of ML deliverables for Provable Resilience Intelligence
# =============================================================
# 
# P2-RISK-MODEL: Span Risk Classification (XGBoost)
#   - Experiment: span_risk_classification
#   - Features: 20 (alarm aggregations, perf metrics, resource attributes)
#   - Target: service_affecting event on logical resource
#   - Metrics: ROC-AUC ~0.99, PR-AUC ~0.95, F1 ~0.92, Recall 1.0
#   - Top predictors: root_cause_count, threshold_crossing_count, minor_alarm_count
#
# P2-ANOMALY-DETECTION: Optical Degradation (Isolation Forest)
#   - Experiment: optical_anomaly_detection
#   - Features: 9 (availability, utilization, error count, indicator values, thresholds)
#   - Contamination: 5%, ~500 anomalies detected
#   - Validation: elevated threshold crossing rate in anomalous readings
#
# P3-RESILIENCE-AGENT: Genie Agent
#   - Agent: 'Lakelink Fiber Resilience Intelligence'
#   - Tables: alarm, resource_performance, logical_resource, equipment,
#     topology_graph, connection_point, customer_facing_service,
#     geographic_site, customer, anomaly
#
# P3-AI-GATEWAY: Unity AI Gateway
#   - Service: cdm_tmforum.provable_resilience_intelligence.resilience-agent-gateway
#   - Primary: Claude Opus 4.5 (pay-per-token)
#   - Fallback: GPT-5-mini
#   - Rate limits: 30 req/min/user, 1000 req/hr service-wide
#   - Inference table: resilience_agent_payload

print("All ML deliverables created successfully.")